In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.pipeline import Pipeline, PipelineModel
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import *
from pyspark.sql.types import *
import mlflow

In [0]:
model_path = "/Volumes/nff_catalog_f586_dev/gold_ml/model/churn.model"

In [0]:
def train_churn_model(table_name = "nff_catalog_f586_dev.gold_ml.churn_features",
                      experiment_name="/Shared/churn_model_experiment",
                      run_name="churn_rf_pipeline"):
    
    pipeline_stages = []

    #TASK 1: CREATE A DATAFRAME
    df = spark.read.table(table_name)
    
    #TASK 2: STRINGS TO NUMBERS
    categorical_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, StringType) and f.name != "is_churned"
    ]
    for c in categorical_cols:
       string_indexer = StringIndexer(inputCol=c, outputCol=c + '_index', handleInvalid="keep")
       onehot_encoder = OneHotEncoder(inputCol=string_indexer.getOutputCol(), outputCol=c + '_vec', dropLast=False)
       pipeline_stages += [string_indexer,onehot_encoder]

    #TASK 3: NUMERICAL FEATURES TO VectorAssembler
    numerical_types = (DoubleType, IntegerType, FloatType, DecimalType, FloatType)
    numerical_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, numerical_types) and f.name != "is_churned"
    ]
    feature_cols = numerical_cols + [c + '_vec' for c in categorical_cols]
    assembler = VectorAssembler(inputCols = feature_cols, outputCol="raw_features")
    pipeline_stages += [assembler]

    #TASK 4: NORMALIZE THE "outputCol" OF VectorAssembler USING StandardScaler
    scaler = StandardScaler(inputCol="raw_features", outputCol="features")
    pipeline_stages += [scaler]

    #TASK 5: DEFINE THE MODEL
    rf = RandomForestClassifier(labelCol="is_churned", featuresCol="features", numTrees=100)
    pipeline_stages += [rf]

    #TASK 6: BUILD THE PIPELINE
    pipeline = Pipeline(stages=pipeline_stages)

    #TASK 7: SPLIT DATAFRAME
    train_data, test_data = df.randomSplit([0.7, 0.3], seed=42)

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_param("table_name", table_name)
        mlflow.log_param("num_trees", rf.getNumTrees())
        mlflow.log_param("train_rows", train_data.count())
        mlflow.log_param("test_rows", test_data.count())
        mlflow.log_param("num_categorical_cols", len(categorical_cols))
        mlflow.log_param("num_numerical_cols", len(numerical_cols))
        
        #TASK 8: TRAIN
        model = pipeline.fit(train_data)

        #TASK 9: PREDICTION
        predictions = model.transform(test_data)

        #TASK 10: EVALUATE
        evaluator = BinaryClassificationEvaluator(labelCol="is_churned", metricName="areaUnderROC")
        accuracy = evaluator.evaluate(predictions)
        print(f"Accuracy = {accuracy}")

        print("-" * 30)
        print(f"✅ Model Training Complete!")
        print(f"📊 Area Under ROC (Accuracy): {accuracy:.4f}")
        print("-" * 30)

        mlflow.log_param("auc", accuracy)
        mlflow.spark.log_model(model,artifact_path="churn_model")

        #TASK 11: SAVE MODEL
        #model.write().overwrite().save("/dbfs/FileStore/models/churn_model

In [0]:
model = train_churn_model()